# Baladiya Concierge — Bilingual Classifier Training

Retrains the civic intent classifier on the full bilingual dataset (EN + MSA + Lebanese + Arabizi).
Reports per-variety F1 on the held-out test set.

**Run after**: `python3 build_dataset.md && python3 dataset_english.md`  
**Artifact**: `modelserver/artifacts/classifier.joblib`

In [1]:
import pandas as pd
import numpy as np
import joblib
import hashlib
import json
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

CSV_PATH = Path('../civic_intent_dataset.csv')
ARTIFACT_PATH = Path('../modelserver/artifacts/classifier.joblib')
RESULTS_PATH = Path('../evals/classifier_bilingual_results.json')

df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df)}')
print(df.groupby(['lang','variety','split']).size().unstack(fill_value=0))

Total rows: 13083
split          test  train
lang variety              
ar   arabizi     86    314
     lebanese    33    179
     msa         69    246
en   en        2425   9731


In [2]:
# Train / test split (deterministic — already encoded in CSV)
train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()
print(f'Train: {len(train_df)}  Test: {len(test_df)}')
print('Test intent dist:', test_df['intent'].value_counts().to_dict())
print('Test variety dist:', test_df['variety'].value_counts().to_dict())

Train: 10470  Test: 2613
Test intent dist: {'question': 680, 'spam': 676, 'human': 674, 'report': 583}
Test variety dist: {'en': 2425, 'arabizi': 86, 'msa': 69, 'lebanese': 33}


In [3]:
# Pipeline — char n-grams (3-5) + word unigrams handle Arabizi/Lebanese spelling variation
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=150_000,
        sublinear_tf=True,
        strip_accents=None,
        lowercase=True,
    )),
    ('clf', LogisticRegression(
        C=5.0,
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
    )),
])

pipeline.fit(train_df['text'], train_df['intent'])
print('Training complete.')

Training complete.


In [4]:
# Overall evaluation
y_pred = pipeline.predict(test_df['text'])
macro_f1 = f1_score(test_df['intent'], y_pred, average='macro')
print(f'Overall macro-F1: {macro_f1:.4f}')
print()
print(classification_report(test_df['intent'], y_pred))

Overall macro-F1: 0.9973

              precision    recall  f1-score   support

       human       1.00      1.00      1.00       674
    question       1.00      0.99      1.00       680
      report       0.99      1.00      1.00       583
        spam       1.00      1.00      1.00       676

    accuracy                           1.00      2613
   macro avg       1.00      1.00      1.00      2613
weighted avg       1.00      1.00      1.00      2613



In [5]:
# Per-language F1
results = {'overall_macro_f1': round(macro_f1, 4), 'per_lang': {}, 'per_variety': {}}

for lang in ['en', 'ar']:
    mask = test_df['lang'] == lang
    if mask.sum() == 0:
        continue
    f1 = f1_score(test_df.loc[mask,'intent'], pipeline.predict(test_df.loc[mask,'text']), average='macro')
    results['per_lang'][lang] = round(f1, 4)
    print(f'lang={lang}  macro-F1={f1:.4f}  (n={mask.sum()})')

print()

# Per-variety F1
for var in ['en', 'msa', 'lebanese', 'arabizi']:
    mask = test_df['variety'] == var
    if mask.sum() == 0:
        print(f'variety={var}  NO TEST ROWS')
        continue
    f1 = f1_score(test_df.loc[mask,'intent'], pipeline.predict(test_df.loc[mask,'text']), average='macro')
    results['per_variety'][var] = round(f1, 4)
    print(f'variety={var:10}  macro-F1={f1:.4f}  (n={mask.sum()})')

lang=en  macro-F1=0.9984  (n=2425)
lang=ar  macro-F1=0.9798  (n=188)



variety=en          macro-F1=0.9984  (n=2425)
variety=msa         macro-F1=1.0000  (n=69)
variety=lebanese    macro-F1=1.0000  (n=33)
variety=arabizi     macro-F1=0.9636  (n=86)


In [6]:
# Save artifact
ARTIFACT_PATH.parent.mkdir(exist_ok=True)
joblib.dump(pipeline, ARTIFACT_PATH)

# SHA-256
sha256 = hashlib.sha256(ARTIFACT_PATH.read_bytes()).hexdigest()
data_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

results['artifact_sha256'] = sha256
results['data_sha256'] = data_sha256
results['total_rows'] = len(df)
results['train_rows'] = len(train_df)
results['test_rows'] = len(test_df)

RESULTS_PATH.parent.mkdir(exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2))

print(f'Artifact saved: {ARTIFACT_PATH}')
print(f'Artifact SHA-256: {sha256}')
print(f'Data SHA-256:     {data_sha256}')
print(f'Results saved: {RESULTS_PATH}')
print()
print('=== SUMMARY ===')
print(f'  Overall macro-F1:  {results["overall_macro_f1"]}')
print(f'  EN macro-F1:       {results["per_lang"].get("en", "N/A")}')
print(f'  AR macro-F1:       {results["per_lang"].get("ar", "N/A")}')
print(f'  MSA F1:            {results["per_variety"].get("msa", "N/A")}')
print(f'  Lebanese F1:       {results["per_variety"].get("lebanese", "N/A")}')
print(f'  Arabizi F1:        {results["per_variety"].get("arabizi", "N/A")}')

Artifact saved: ../modelserver/artifacts/classifier.joblib
Artifact SHA-256: bd2d33060edcc9c7e02246fa6b499174928df9875474abd32c2967ef0c1edc0d
Data SHA-256:     f225e547ddb29575bc380375a50879b515f556786be1ce9eb1b56922498dff4e
Results saved: ../evals/classifier_bilingual_results.json

=== SUMMARY ===
  Overall macro-F1:  0.9973
  EN macro-F1:       0.9984
  AR macro-F1:       0.9798
  MSA F1:            1.0
  Lebanese F1:       1.0
  Arabizi F1:        0.9636
